# 受試者評分排序分析 (Subject Ratings Sorting Analysis)

本筆記本將載入 `sub-01`、`sub-02` 與 `sub-03` 的 fMRI 評分資料，並分別根據其 **Grasp Rating (抓取評分)** 與 **Hold Rating (握持評分)** 進行排序。

In [12]:
import pandas as pd
import os
import numpy as np
from pathlib import Path

# 定義檔案路徑
data_dir = r'D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii'
base_dir = r'D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject'
subjects = ['sub-01', 'sub-02', 'sub-03']

# 載入受試者資料
dfs = {}
for sub in subjects:
    file_path = os.path.join(data_dir, f'{sub}_condition_with_ratings.csv')
    dfs[sub] = pd.read_csv(file_path)
    print(f'已載入 {sub} 資料，共 {len(dfs[sub])} 筆資料。')

已載入 sub-01 資料，共 8640 筆資料。
已載入 sub-02 資料，共 8640 筆資料。
已載入 sub-03 資料，共 8640 筆資料。


## 1. 按照 Grasp Rating (抓取評分) 排序

我們將各受試者的資料依據 `grasp_rating` 由大到小（降序）進行排序，展示評分最高的前 10 張照片，並將完整的排序結果存檔。

In [14]:
import os
import pandas as pd

subjects = ['sub-01', 'sub-02', 'sub-03']

dfs = {}

for sub in subjects:

    # 讀取
    file_path = os.path.join(
        data_dir,
        f'{sub}_condition_with_ratings.csv'
    )

    df = pd.read_csv(file_path)

    print(f'已載入 {sub} 資料，共 {len(df)} 筆資料')

    # grasp排序（高→低）
    df_sorted = (
        df
        .sort_values(
            by=[
                "grasp_rating",
                "concept",
                "session",
                "run",
                "trial_idx"
            ],
            ascending=[False, True, True, True, True]
        )
        .reset_index(drop=True)
    )

    # 存進字典
    dfs[sub] = df_sorted

    # 儲存
    save_path = os.path.join(
        data_dir,
        f'{sub}_condition_sort_by_grasp.csv'
    )

    df_sorted.to_csv(save_path, index=False)

    print(f'已儲存 {save_path}')

已載入 sub-01 資料，共 8640 筆資料
已儲存 D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii\sub-01_condition_sort_by_grasp.csv
已載入 sub-02 資料，共 8640 筆資料
已儲存 D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii\sub-02_condition_sort_by_grasp.csv
已載入 sub-03 資料，共 8640 筆資料
已儲存 D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii\sub-03_condition_sort_by_grasp.csv


## 2. 按照 Hold Rating (握持評分) 排序

我們將各受試者的資料依據 `hold_rating` 由大到小（降序）進行排序，展示評分最高的前 10 張照片，並將完整的排序結果存檔。

In [5]:
print('=== 按照 hold_rating 排序 (由高到低前 10 名) ===\n')

subjects = ['sub-01', 'sub-02', 'sub-03']

dfs = {}

for sub in subjects:

    # 讀取
    file_path = os.path.join(
        data_dir,
        f'{sub}_condition_with_ratings.csv'
    )

    df = pd.read_csv(file_path)

    print(f'已載入 {sub} 資料，共 {len(df)} 筆資料')

    # grasp排序（高→低）
    df_sorted = (
        df
        .sort_values(
            by=[
                "hold_rating",
                "concept",
                "session",
                "run",
                "trial_idx"
            ],
            ascending=[False, True, True, True, True]
        )
        .reset_index(drop=True)
    )

    # 存進字典
    dfs[sub] = df_sorted

    # 儲存
    save_path = os.path.join(
        data_dir,
        f'{sub}_condition_sort_by_hold.csv'
    )

    df_sorted.to_csv(save_path, index=False)

    print(f'已儲存 {save_path}')

=== 按照 hold_rating 排序 (由高到低前 10 名) ===

已載入 sub-01 資料，共 8640 筆資料
已儲存 D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii\sub-01_condition_sort_by_hold.csv
已載入 sub-02 資料，共 8640 筆資料
已儲存 D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii\sub-02_condition_sort_by_hold.csv
已載入 sub-03 資料，共 8640 筆資料
已儲存 D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\subject_fMRI_nii\sub-03_condition_sort_by_hold.csv


In [15]:
base_dir = Path(r"D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject")

sub01_grasp_sort = pd.read_csv(
    base_dir / "subject_fMRI_nii\sub-01_condition_sort_by_grasp.csv"
)

print(sub01_grasp_sort.columns)

Index(['image_filename', 'session', 'run', 'trial_idx', 'subject', 'concept',
       'grasp_rating', 'hold_rating'],
      dtype='object')


In [20]:
import contextlib
import nibabel as nib

def compute_concept_average(
    df_sorted,
    base_dir,
    sub_id="sub-01",
    verbose=True
):
    """
    Compute concept-averaged fMRI volumes while preserving
    concept order in df_sorted.

    Returns
    -------
    concept_volumes
        shape = (72,91,75,n_concepts)

    concept_names
        list of concepts in matching order
    """

    concept_volumes = []
    concept_names = []

    # 保持 dataframe 順序
    unique_concepts = (
        df_sorted["concept"]
        .drop_duplicates()
        .tolist()
    )

    for concept in unique_concepts:

        df_concept = df_sorted[
            df_sorted["concept"] == concept
        ]

        vols = []

        for row in df_concept.itertuples():

            img_path = (
                base_dir /
                "subject_fMRI_nii" /
                sub_id /
                row.session /
                f"{sub_id}_{row.session}_run-{row.run:02d}_betas.nii"
            )

            img = nib.load(img_path)
            data = img.get_fdata()

            vol = data[..., int(row.trial_idx)]

            vols.append(vol)

        mean_vol = np.mean(vols, axis=0)

        concept_volumes.append(mean_vol)
        concept_names.append(concept)

    concept_volumes = np.stack(
        concept_volumes,
        axis=-1
    )

    if verbose:
        print(
            "\nFinal shape:",
            concept_volumes.shape
        )
    

    return concept_volumes, concept_names

In [9]:
df_grasp_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-01_condition_sort_by_grasp.csv")

sub01_grasp_volumes, sub01_grasp_names = compute_concept_average(
    df_grasp_sorted,
    base_dir,
    sub_id="sub-01",
    verbose=True
)
print("===sub01 grasp done===\n")


Final shape: (72, 91, 75, 720)
Saved: D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\processed_data\sub-01_concept_average.nii.gz
===sub01 grasp done===



In [10]:
df_hold_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-01_condition_sort_by_hold.csv")

sub01_hold_volumes, sub01_hold_names = compute_concept_average(
    df_hold_sorted,
    base_dir,
    sub_id="sub-01",
    verbose=True
)
print("===sub01 hold done===\n")


Final shape: (72, 91, 75, 720)
Saved: D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject\processed_data\sub-01_concept_average.nii.gz
===sub01 hold done===



In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

out_dir = base_dir / "processed_data"
out_dir.mkdir(parents=True, exist_ok=True)

np.save(
    out_dir / "sub-01_grasp_volumes.npy",
    sub01_grasp_volumes.astype(np.float32)
)

np.save(
    out_dir / "sub-01_hold_volumes.npy",
    sub01_hold_volumes.astype(np.float32)
)

pd.Series(sub01_grasp_names).to_csv(
    out_dir / "sub-01_grasp_names.csv",
    index=False,
    header=["concept"]
)

pd.Series(sub01_hold_names).to_csv(
    out_dir / "sub-01_hold_names.csv",
    index=False,
    header=["concept"]
)

In [21]:
df_grasp_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-02_condition_sort_by_grasp.csv")

sub02_grasp_volumes, sub02_grasp_names = compute_concept_average(
    df_grasp_sorted,
    base_dir,
    sub_id="sub-02",
    verbose=True
)
print("===sub02 grasp done===\n")

df_hold_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-02_condition_sort_by_hold.csv")

sub02_hold_volumes, sub02_hold_names = compute_concept_average(
    df_hold_sorted,
    base_dir,
    sub_id="sub-02",
    verbose=True
)
print("===sub02 hold done===\n")

np.save(
    out_dir / "sub-02_grasp_volumes.npy",
    sub02_grasp_volumes.astype(np.float32)
)

np.save(
    out_dir / "sub-02_hold_volumes.npy",
    sub02_hold_volumes.astype(np.float32)
)

pd.Series(sub02_grasp_names).to_csv(
    out_dir / "sub-02_grasp_names.csv",
    index=False,
    header=["concept"]
)

pd.Series(sub02_hold_names).to_csv(
    out_dir / "sub-02_hold_names.csv",
    index=False,
    header=["concept"]
)


Final shape: (72, 94, 75, 720)
===sub02 grasp done===


Final shape: (72, 94, 75, 720)
===sub02 hold done===



In [22]:
df_grasp_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-03_condition_sort_by_grasp.csv")

sub03_grasp_volumes, sub03_grasp_names = compute_concept_average(
    df_grasp_sorted,
    base_dir,
    sub_id="sub-03",
    verbose=True
)
print("===sub03 grasp done===\n")

df_hold_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-03_condition_sort_by_hold.csv")

sub03_hold_volumes, sub03_hold_names = compute_concept_average(
    df_hold_sorted,
    base_dir,
    sub_id="sub-03",
    verbose=True
)
print("===sub03 hold done===\n")

np.save(
    out_dir / "sub-03_grasp_volumes.npy",
    sub03_grasp_volumes.astype(np.float32)
)

np.save(
    out_dir / "sub-03_hold_volumes.npy",
    sub03_hold_volumes.astype(np.float32)
)

pd.Series(sub03_grasp_names).to_csv(
    out_dir / "sub-03_grasp_names.csv",
    index=False,
    header=["concept"]
)

pd.Series(sub03_hold_names).to_csv(
    out_dir / "sub-03_hold_names.csv",
    index=False,
    header=["concept"]
)


Final shape: (71, 83, 73, 720)
===sub03 grasp done===


Final shape: (71, 83, 73, 720)
===sub03 hold done===



In [11]:
print(len(sub01_grasp_names))
print(sub01_grasp_volumes.shape[-1])

print(len(sub01_hold_names))
print(sub01_hold_volumes.shape[-1])

720
720
720
720


In [37]:
from collections import Counter

print("grasp duplicates:", [k for k, v in Counter(sub01_grasp_names).items() if v > 1])
print("hold duplicates:", [k for k, v in Counter(sub01_hold_names).items() if v > 1])

grasp duplicates: []
hold duplicates: []


In [49]:
grasp = pd.read_csv(base_dir / "subject_fMRI_nii\concepts_sorted_by_grasp.csv")
hold = pd.read_csv(base_dir / "subject_fMRI_nii\concepts_sorted_by_hold.csv")

print(sub01_grasp_names[:10],"\n")
print(sub01_hold_names[:10],"\n")
print(grasp[:10],"\n")
print(hold[:10])

['cashew', 'spoon', 'pom-pom', 'mitten', 'candy_bar', 'comb', 'napkin_ring', 'pencil_sharpener', 'banana', 'toilet_paper'] 

['spoon', 'comb', 'candy_bar', 'peanut', 'shower_cap', 'mitten', 'pencil_sharpener', 'pom-pom', 'key', 'lime'] 

            concept  grasp_rating  hold_rating
0             spoon        6.8378       6.8919
1            cashew        6.8378       6.8378
2           pom-pom        6.8250       6.8500
3            mitten        6.8000       6.8571
4         candy_bar        6.7692       6.8718
5              comb        6.7500       6.8889
6       napkin_ring        6.7500       6.8056
7  pencil_sharpener        6.7429       6.8571
8            banana        6.7368       6.8158
9      toilet_paper        6.7297       6.7568 

            concept  grasp_rating  hold_rating
0             spoon        6.8378       6.8919
1              comb        6.7500       6.8889
2         candy_bar        6.7692       6.8718
3            peanut        6.7179       6.8718
4       

In [40]:
import numpy as np

print("grasp mean:", np.mean(sub01_grasp_volumes))
print("hold mean:", np.mean(sub01_hold_volumes))

print(sub01_grasp_volumes[..., 0].mean())
print(sub01_grasp_volumes[..., 0].std())

grasp mean: 0.000639996895608957
hold mean: 0.0006399968956089587
0.0011873421326995333
0.013018425927856184


In [41]:
unique, counts = np.unique(sub01_grasp_names, return_counts=True)
print(dict(zip(unique, counts)))

{np.str_('acorn'): np.int64(1), np.str_('airbag'): np.int64(1), np.str_('aircraft_carrier'): np.int64(1), np.str_('airplane'): np.int64(1), np.str_('alligator'): np.int64(1), np.str_('aloe'): np.int64(1), np.str_('altar'): np.int64(1), np.str_('aluminum_foil'): np.int64(1), np.str_('anchor'): np.int64(1), np.str_('anklet'): np.int64(1), np.str_('ant'): np.int64(1), np.str_('anteater'): np.int64(1), np.str_('antelope'): np.int64(1), np.str_('antenna'): np.int64(1), np.str_('anvil'): np.int64(1), np.str_('apple'): np.int64(1), np.str_('applesauce'): np.int64(1), np.str_('artichoke'): np.int64(1), np.str_('ashtray'): np.int64(1), np.str_('asparagus'): np.int64(1), np.str_('avocado'): np.int64(1), np.str_('axe'): np.int64(1), np.str_('baby'): np.int64(1), np.str_('backpack'): np.int64(1), np.str_('bag'): np.int64(1), np.str_('bagel'): np.int64(1), np.str_('ball'): np.int64(1), np.str_('balloon'): np.int64(1), np.str_('bamboo'): np.int64(1), np.str_('banana'): np.int64(1), np.str_('banana_s

In [43]:
print(sub01_grasp_volumes.shape)
print(sub01_hold_volumes.shape)

(72, 91, 75, 720)
(72, 91, 75, 720)


In [1]:
df_grasp_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-01_condition_sort_by_grasp.csv")

sub02_grasp_volumes, sub02_grasp_names = compute_concept_average(
    df_grasp_sorted,
    base_dir,
    sub_id="sub-02",
    verbose=True
)
print("===sub02 grasp done===\n")

sub03_grasp_volumes, sub03_grasp_names = compute_concept_average(
    df_grasp_sorted,
    base_dir,
    sub_id="sub-03",
    verbose=True
)
print("===sub03 grasp done===\n")


NameError: name 'pd' is not defined

In [ ]:
df_hold_sorted = pd.read_csv(base_dir / "subject_fMRI_nii\sub-01_condition_sort_by_hold.csv")

sub02_hold_volumes, sub02_hold_names = compute_concept_average(
    df_hold_sorted,
    base_dir,
    sub_id="sub-02",
    verbose=True
)
print("===sub02 hold done===\n")

sub03_hold_volumes, sub03_hold_names = compute_concept_average(
    df_hold_sorted,
    base_dir,
    sub_id="sub-03",
    verbose=True
)
print("===sub03 hold done===\n")
